# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
# import tensorflow_hub as hub

# Load the Dataset

In [ ]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# Clean the dataset and encode labels

In [ ]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


# Split Data and convert to NumPy arrays

In [ ]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

# X_train.shape, X_test.shape, y_train.shape, y_test.shape

# Compute text Statistics for Vectorization

In [ ]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# Create the TextVectorization layer

In [ ]:
from tensorflow.keras.layers import TextVectorization # convert raw text into integer sequences
text_vectorizer = TextVectorization(
    max_tokens=totalWordsLength, # Limits the vocabulary size to total_words_length unique words
    standardize="lower_and_strip_punctuation", # Standardizes text by lowercasing and stripping punctuation
    output_mode="int",
    output_sequence_length=averageWordsLength, # Pads or truncates each sequence to avg_words_len tokens
    # split="whitespace",
)
text_vectorizer.adapt(X_train) # Adapts the layer on training data to learn the vocabulary distribution
# text_vectorizer.get_vocabulary()


# Create the Embedding layer

In [ ]:
embedding_layer = layers.Embedding(
    input_dim=totalWordsLength,
    output_dim=128,
    embeddings_initializer='uniform',
    input_length=averageWordsLength
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# Building Dense embedding model

In [ ]:
input_layer = layers.Input(shape=(1,), dtype=tf.string) # input layer that expects string tensor
vec_layer = text_vectorizer(input_layer) # convert text into token IDs (frequency count)
embedding_layer_model = embedding_layer(vec_layer) # learn dense vector representations for each word

x = layers.GlobalAveragePooling1D()(embedding_layer_model) # compress each sequence into a single vector
# x = layers.Flatten()(x)
x = layers.Dense(32, activation='relu')(x) # Adds a dense hidden layer with ReLU activation for non-linearity

output_layer = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs=input_layer, outputs=output_layer)

model.compile(loss="binary_crossentropy",
              optimizer='adam',
            #   loss=keras.losses.BinaryCrossentropy(label_smoothing=0.5),
              metrics=['accuracy']
              )
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 16)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 16, 128)        │     2,007,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,011,969 (7.68 MB)

 Trainable params: 2,011,969 (7.68 MB)

 Non-trainable params: 0 (0.00 B)

# Training and Evaluating

In [ ]:
history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), validation_steps=int(0.2 * len(X_test)))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8681 - loss: 0.3803 - val_accuracy: 0.9596 - val_loss: 0.1367
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9781 - loss: 0.1028 - val_accuracy: 0.9767 - val_loss: 0.0833
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9885 - loss: 0.0467 - val_accuracy: 0.9794 - val_loss: 0.0695
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9948 - loss: 0.0262 - val_accuracy: 0.9785 - val_loss: 0.0668
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.9981 - loss: 0.0112 - val_accuracy: 0.9785 - val_loss: 0.0637


In [29]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

# Convert the list of messages to a TensorFlow constant of strings
test_messages_tf = tf.constant(test_messages, dtype=tf.string)

preds = model.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    label = "Spam" if p[0] >= 0.4 else "Ham"
    print(f"{label} → {msg} ({p[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Ham → Hey, are we meeting today? (0.00)
Spam → WIN cash now!!! Click the link (0.73)
Ham → Your OTP is 348921 (0.08)
Spam → You have won $1000! (0.42)
Spam → Congratulations! You have won a free lottery ticket (0.93)
Ham → Don't forget to submit the assignment (0.05)
Spam → URGENT! You won a cash prize. Call immediately! (1.00)
Ham → My friend won a prize yesterday (0.25)
